In [19]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

encoder_epoch_df = pd.read_hdf(
    Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
    key="encoder/prepost_1s_speedThresh_1cms"
)

from src.qc.qc_events import load_behavior_qc_tables

events_df, session_summary_df, _ = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 29)
Valid windows: (83023, 29)
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [2]:
%connect_info

{"key":"8fc4457c-4483-4ec3-b55c-b08e900f0984","signature_scheme":"hmac-sha256","transport":"tcp","ip":"127.0.0.1","hb_port":9000,"control_port":9001,"shell_port":9002,"stdin_port":9003,"iopub_port":9004,"kernel_name":"python3820jvsc74a57bd07f6243129cfff82ede3c5c59589229000c07b9cd1160ea9b86b0dd361ed08050"}

Paste the above JSON into a file, and connect with:
    $> jupyter <app> --existing <file>
or, if you are local, you can connect with just:
    $> jupyter <app> --existing /run/user/1000/jupyter/runtime/kernel-v365c51bbedcb8a1eb99d32767840a52be41f02e01.json
or even just:
    $> jupyter <app> --existing
if this is the most recent Jupyter kernel you have started.


In [20]:
from src.proc.behavior_metrics import add_normalized_phase_day

encoder_epoch_df = add_normalized_phase_day(encoder_epoch_df)
encoder_epoch_df.head()

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,net_direction_bias,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net_movement,dominant_locomotor_state,phase_session_number,n_sessions_in_phase,normalized_phase_day
0,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,post,1.0,air_off_mid_post_1s,...,0.680638,0.0808,0.9192,0.6118,0.3074,0.0,forward,1.0,15.0,0.0
1,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,pre,1.0,air_off_mid_pre_1s,...,0.755565,0.3230,0.6770,0.4804,0.1966,0.0,forward,1.0,15.0,0.0
2,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,post,1.0,air_off_post_1s,...,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward,1.0,15.0,0.0
3,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,pre,1.0,air_off_pre_1s,...,0.951624,0.0340,0.9660,0.8530,0.1130,0.0,forward,1.0,15.0,0.0
4,NML_04,2026_01_12,air_training,1,air_on,main,19.9564,post,1.0,air_on_post_1s,...,-0.487133,0.5560,0.4440,0.0924,0.3516,0.0,stationary,1.0,15.0,0.0


In [29]:
from src.utils.pdata_organize import make_session_availability_summary

session_availability_df = make_session_availability_summary(
    events_df=events_df,
    windows_df=windows_df,
    min_session_duration_s=900,
    min_valid_events=3,
)

In [27]:
session_availability_df.sort_values(
    ["phase", "animal", "date"]
)

,animal,date,phase,session_duration_s,n_events_total,n_events_valid,short_recording,enough_valid_events,good_session_basic,nwin_LED_off_post,...,nwin_air_on_mid_pre,nwin_pseudo_tone_off_post,nwin_pseudo_tone_off_pre,nwin_pseudo_tone_on_post,nwin_pseudo_tone_on_pre,nwin_tone_off_post,nwin_tone_off_pre,nwin_tone_on_post,nwin_tone_on_pre,n_valid_windows_total
16,NML_04,2026_01_12,air_training,1308.9,59,57,False,True,True,0,...,16,17,25,57,57,0,0,0,0,488
17,NML_04,2026_01_13,air_training,1307.0,55,53,False,True,True,0,...,14,14,18,53,53,0,0,0,0,418
18,NML_04,2026_01_14,air_training,1506.2,59,57,False,True,True,0,...,27,27,38,57,57,0,0,0,0,565
19,NML_04,2026_01_15,air_training,1319.9,60,58,False,True,True,0,...,8,8,27,58,58,0,0,0,0,513
20,NML_04,2026_01_16,air_training,1354.6,64,62,False,True,True,0,...,1,1,3,62,62,0,0,0,0,408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,NML_08,2026_03_17,tone_air_training,1235.7,48,46,False,True,True,0,...,26,0,0,0,0,26,40,46,46,492
209,NML_08,2026_03_18,tone_air_training,1236.0,51,49,False,True,True,0,...,0,0,0,0,0,0,39,49,49,462
210,NML_08,2026_03_19,tone_air_training,1214.1,50,48,False,True,True,0,...,1,0,0,0,0,1,37,48,48,445
211,NML_08,2026_03_20,tone_air_training,1220.1,48,46,False,True,True,0,...,20,0,0,0,0,20,39,46,46,476


In [30]:
phase_animal_summary = (
    session_availability_df
    .groupby(["phase", "animal"], dropna=False)
    .agg(
        n_sessions=("date", "nunique"),
        n_good_sessions=("good_session_basic", "sum"),
        n_short_sessions=("short_recording", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

phase_animal_summary

,phase,animal,n_sessions,n_good_sessions,n_short_sessions,first_date,last_date
0,air_training,NML_04,15,15,0,2026_01_12,2026_01_27
1,air_training,NML_05,15,14,0,2026_01_12,2026_01_27
2,air_training,NML_06,15,15,0,2026_01_12,2026_01_27
3,air_training,NML_07,16,16,0,2026_02_21,2026_03_11
4,air_training,NML_08,17,16,1,2026_02_21,2026_03_11
5,habituation,NML_04,16,16,0,2025_12_27,2026_01_11
6,habituation,NML_05,16,16,0,2025_12_27,2026_01_11
7,habituation,NML_06,16,16,0,2025_12_27,2026_01_11
8,habituation,NML_07,19,14,0,2026_02_02,2026_02_20
9,habituation,NML_08,18,15,0,2026_02_02,2026_02_20


In [36]:
from src.utils.pdata_organize import add_day_bins_to_sessions

session_day_df = add_day_bins_to_sessions(
    session_availability_df,
    use_good_sessions_only=True,
)

In [32]:
cols = [
    "phase",
    "animal",
    "date",
    "good_session_basic",
    "session_duration_s",
    "n_events_valid",
    "phase_day_number_good",
    "n_good_sessions_in_phase",
    "normalized_phase_day_good",
    "phase_day_bin",
]

session_day_df[cols].sort_values(
    ["phase", "animal", "date"]
)

,phase,animal,date,good_session_basic,session_duration_s,n_events_valid,phase_day_number_good,n_good_sessions_in_phase,normalized_phase_day_good,phase_day_bin
16,air_training,NML_04,2026_01_12,True,1308.9,57,1.0,15.0,0.000000,early
17,air_training,NML_04,2026_01_13,True,1307.0,53,2.0,15.0,0.071429,early
18,air_training,NML_04,2026_01_14,True,1506.2,57,3.0,15.0,0.142857,early
19,air_training,NML_04,2026_01_15,True,1319.9,58,4.0,15.0,0.214286,early
20,air_training,NML_04,2026_01_16,True,1354.6,62,5.0,15.0,0.285714,early
...,...,...,...,...,...,...,...,...,...,...
208,tone_air_training,NML_08,2026_03_17,True,1235.7,46,6.0,10.0,0.555556,middle
209,tone_air_training,NML_08,2026_03_18,True,1236.0,49,7.0,10.0,0.666667,middle
210,tone_air_training,NML_08,2026_03_19,True,1214.1,48,8.0,10.0,0.777778,late
211,tone_air_training,NML_08,2026_03_20,True,1220.1,46,9.0,10.0,0.888889,late


In [37]:
session_epoch_df = session_epoch_df.merge(
    session_day_df[
        [
            "animal",
            "date",
            "phase",
            "phase_day_number_good",
            "n_good_sessions_in_phase",
            "normalized_phase_day_good",
            "phase_day_bin",
            "good_session_basic",
        ]
    ],
    on=["animal", "date", "phase"],
    how="left"
)



In [38]:
session_epoch_good_df = session_epoch_df[
    session_epoch_df["good_session_basic"] == True
].copy()

In [39]:

from src.utils.pdata_organize import build_validated_epoch_matrix


epoch_matrix_df = build_validated_epoch_matrix(
    encoder_epoch_df=encoder_epoch_df,
    session_day_df=session_day_df,
    keep_good_sessions_only=True,
    keep_valid_windows_only=True,
)

In [40]:
epoch_matrix_df[
    [
        "phase",
        "animal",
        "date",
        "good_session_basic",
        "phase_day_number_good",
        "normalized_phase_day_good",
        "phase_day_bin",
        "session_time_bin",
        "event_number",
        "anchor_name",
        "window_position",
        "epoch_name",
    ]
].head(30)

,phase,animal,date,good_session_basic,phase_day_number_good,normalized_phase_day_good,phase_day_bin,session_time_bin,event_number,anchor_name,window_position,epoch_name
0,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,pseudo_tone_on,post,pseudo_tone_on_post_1s
1,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,pseudo_tone_on,pre,pseudo_tone_on_pre_1s
2,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_on,post,air_on_post_1s
3,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_on,pre,air_on_pre_1s
4,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_off,post,air_off_post_1s
5,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_off,pre,air_off_pre_1s
6,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_off_mid,post,air_off_mid_post_1s
7,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,1,air_off_mid,pre,air_off_mid_pre_1s
8,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,2,pseudo_tone_on,post,pseudo_tone_on_post_1s
9,air_training,NML_04,2026_01_12,True,1.0,0.0,early,early,2,pseudo_tone_on,pre,pseudo_tone_on_pre_1s


In [42]:
epoch_matrix_df.groupby(
    ["phase", "animal", "phase_day_bin"],
    dropna=False
).agg(
    n_dates=("date", "nunique"),
    n_rows=("epoch_name", "count"),
    n_events=("event_number", "nunique"),
).reset_index()

,phase,animal,phase_day_bin,n_dates,n_rows,n_events
0,air_training,NML_04,early,5,2392,62
1,air_training,NML_04,late,5,2582,50
2,air_training,NML_04,middle,5,2470,52
3,air_training,NML_05,early,5,2304,64
4,air_training,NML_05,late,4,448,17
5,air_training,NML_05,middle,5,926,36
6,air_training,NML_06,early,5,2127,55
7,air_training,NML_06,late,5,1663,49
8,air_training,NML_06,middle,5,2618,52
9,air_training,NML_07,early,6,2252,71


In [43]:
from statsmodels.stats.anova import AnovaRM
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Select tone-on windows during tone-air training
# --------------------------------------------------

tone_on_df = epoch_matrix_df[
    (epoch_matrix_df["phase"] == "tone_air_training") &
    (epoch_matrix_df["anchor_name"] == "tone_on") &
    (epoch_matrix_df["phase_day_bin"].isin(["early", "middle", "late"])) &
    (epoch_matrix_df["window_position"].isin(["pre", "post"]))
].copy()

# --------------------------------------------------
# 2. Average to animal × day-bin × pre/post level
# --------------------------------------------------
# This averages across sessions, events, and session_time_bin for now.

tone_on_rm_df = (
    tone_on_df
    .groupby(
        ["animal", "phase_day_bin", "window_position"],
        dropna=False
    )
    .agg(
        frac_forward=("frac_forward", "mean"),
        n_windows=("frac_forward", "count"),
        n_events=("event_number", "nunique"),
        n_dates=("date", "nunique"),
    )
    .reset_index()
)

# Optional: enforce categorical order
tone_on_rm_df["phase_day_bin"] = pd.Categorical(
    tone_on_rm_df["phase_day_bin"],
    categories=["early", "middle", "late"],
    ordered=True
)

tone_on_rm_df["window_position"] = pd.Categorical(
    tone_on_rm_df["window_position"],
    categories=["pre", "post"],
    ordered=True
)

tone_on_rm_df

,animal,phase_day_bin,window_position,frac_forward,n_windows,n_events,n_dates
0,NML_04,early,post,0.225380,143,38,4
1,NML_04,early,pre,0.158259,143,38,4
2,NML_04,late,post,0.222479,71,53,3
3,NML_04,late,pre,0.169141,71,53,3
4,NML_04,middle,post,0.233484,76,30,3
5,NML_04,middle,pre,0.120868,76,30,3
6,NML_05,early,post,0.554759,64,23,4
7,NML_05,early,pre,0.324188,64,23,4
8,NML_05,late,post,0.555212,49,20,3
9,NML_05,late,pre,0.308220,49,20,3


In [44]:
cell_check = (
    tone_on_rm_df
    .groupby("animal")
    .size()
    .reset_index(name="n_cells")
)

cell_check

,animal,n_cells
0,NML_04,6
1,NML_05,6
2,NML_06,6
3,NML_07,6
4,NML_08,6


In [46]:
aov = AnovaRM(
    data=tone_on_rm_df,
    depvar="frac_forward",
    subject="animal",
    within=["phase_day_bin", "window_position"]
).fit()

print(aov)

tone_on_rm_df.groupby(["phase_day_bin", "window_position"])["frac_forward"].mean()

                          Anova
                              F Value Num DF Den DF Pr > F
----------------------------------------------------------
phase_day_bin                  0.9454 2.0000 8.0000 0.4280
window_position               16.9333 1.0000 4.0000 0.0147
phase_day_bin:window_position  0.1975 2.0000 8.0000 0.8246



phase_day_bin  window_position
early          pre                0.165872
               post               0.299246
middle         pre                0.185638
               post               0.303817
late           pre                0.149128
               post               0.269313
Name: frac_forward, dtype: float64

In [47]:
from statsmodels.stats.anova import AnovaRM
import pandas as pd

# --------------------------------------------------
# Tone-on speed analysis
# phase = tone_air_training
# anchor = tone_on
# outcome = mean_speed_path_cms
# --------------------------------------------------

tone_on_speed_df = epoch_matrix_df[
    (epoch_matrix_df["phase"] == "tone_air_training") &
    (epoch_matrix_df["anchor_name"] == "tone_on") &
    (epoch_matrix_df["phase_day_bin"].isin(["early", "middle", "late"])) &
    (epoch_matrix_df["window_position"].isin(["pre", "post"]))
].copy()

# Average to animal × day-bin × pre/post level
tone_on_speed_rm_df = (
    tone_on_speed_df
    .groupby(
        ["animal", "phase_day_bin", "window_position"],
        dropna=False
    )
    .agg(
        mean_speed_path_cms=("mean_speed_path_cms", "mean"),
        n_windows=("mean_speed_path_cms", "count"),
        n_events=("event_number", "nunique"),
        n_dates=("date", "nunique"),
    )
    .reset_index()
)

tone_on_speed_rm_df["phase_day_bin"] = pd.Categorical(
    tone_on_speed_rm_df["phase_day_bin"],
    categories=["early", "middle", "late"],
    ordered=True
)

tone_on_speed_rm_df["window_position"] = pd.Categorical(
    tone_on_speed_rm_df["window_position"],
    categories=["pre", "post"],
    ordered=True
)

# Check cells
cell_check_speed = (
    tone_on_speed_rm_df
    .groupby("animal")
    .size()
    .reset_index(name="n_cells")
)

cell_check_speed

,animal,n_cells
0,NML_04,6
1,NML_05,6
2,NML_06,6
3,NML_07,6
4,NML_08,6


In [48]:
aov_speed = AnovaRM(
    data=tone_on_speed_rm_df,
    depvar="mean_speed_path_cms",
    subject="animal",
    within=["phase_day_bin", "window_position"]
).fit()

print(aov_speed)

                          Anova
                              F Value Num DF Den DF Pr > F
----------------------------------------------------------
phase_day_bin                  1.8306 2.0000 8.0000 0.2215
window_position                9.7565 1.0000 4.0000 0.0354
phase_day_bin:window_position  0.0474 2.0000 8.0000 0.9540



In [49]:
speed_means = (
    tone_on_speed_rm_df
    .groupby(["phase_day_bin", "window_position"], observed=False)
    .agg(
        mean_speed=("mean_speed_path_cms", "mean"),
        sem_speed=("mean_speed_path_cms", lambda x: x.std(ddof=1) / (len(x) ** 0.5)),
        n_animals=("animal", "nunique"),
    )
    .reset_index()
)

speed_means

,phase_day_bin,window_position,mean_speed,sem_speed,n_animals
0,early,pre,0.601394,0.207809,5
1,early,post,0.873768,0.268174,5
2,middle,pre,0.686888,0.254414,5
3,middle,post,0.941554,0.302323,5
4,late,pre,0.470968,0.144546,5
5,late,post,0.747228,0.240752,5
